In [2]:
import pandas as pd
meta = pd.read_csv('../data_src/container_meta.csv',header=None, names=['container_id', 'machine_id', 'time_stamp','app_id', 'status', 'cpu_request', 'cpu_limit','mem_size'])
print(meta)

       container_id machine_id  time_stamp    app_id   status  cpu_request  \
0               c_1     m_2556           0  app_5052  started        400.0   
1               c_1     m_2556      287942  app_5052  started        400.0   
2               c_1     m_2556      338909  app_5052  started        400.0   
3               c_2      m_962           0  app_8125  started        800.0   
4               c_2      m_962       23205  app_8125  started        800.0   
...             ...        ...         ...       ...      ...          ...   
109963      c_21224     m_2072           0  app_3288  started        800.0   
109964      c_21224     m_2072       13168  app_3288  started        800.0   
109965      c_21224     m_2072      108476  app_3288  started        800.0   
109966      c_21224     m_2072      177760  app_3288  started        800.0   
109967      c_21224     m_2072      260292  app_3288  started          NaN   

        cpu_limit  mem_size  
0           400.0      1.56  
1  

In [3]:
meta = meta[meta['status']=='started'][['container_id','app_id','cpu_request']]
print(meta)

       container_id    app_id  cpu_request
0               c_1  app_5052        400.0
1               c_1  app_5052        400.0
2               c_1  app_5052        400.0
3               c_2  app_8125        800.0
4               c_2  app_8125        800.0
...             ...       ...          ...
109963      c_21224  app_3288        800.0
109964      c_21224  app_3288        800.0
109965      c_21224  app_3288        800.0
109966      c_21224  app_3288        800.0
109967      c_21224  app_3288          NaN

[109293 rows x 3 columns]


In [4]:
meta.dropna(subset=['cpu_request'], inplace=True)
print(meta)

       container_id    app_id  cpu_request
0               c_1  app_5052        400.0
1               c_1  app_5052        400.0
2               c_1  app_5052        400.0
3               c_2  app_8125        800.0
4               c_2  app_8125        800.0
...             ...       ...          ...
109962      c_21223  app_3514        400.0
109963      c_21224  app_3288        800.0
109964      c_21224  app_3288        800.0
109965      c_21224  app_3288        800.0
109966      c_21224  app_3288        800.0

[109292 rows x 3 columns]


In [5]:
meta.drop_duplicates(subset=['container_id','app_id'], keep='first',inplace=True)
print(meta)

       container_id    app_id  cpu_request
0               c_1  app_5052        400.0
3               c_2  app_8125        800.0
8               c_3    app_66        400.0
13              c_4  app_3222        400.0
20              c_5  app_5955        400.0
...             ...       ...          ...
109940      c_21219  app_3387        400.0
109953      c_21221  app_4799        800.0
109956      c_21222  app_7604        400.0
109959      c_21223  app_3514        400.0
109963      c_21224  app_3288        800.0

[21103 rows x 3 columns]


In [11]:
meta.sort_values(by='app_id', ascending=True,inplace=True)
print(meta,meta['cpu_request'].max())

       container_id   app_id  cpu_request
81915       c_15794    app_1        400.0
80794       c_15576   app_10        800.0
101468      c_19567   app_10        800.0
715           c_136   app_10        800.0
50868        c_9842   app_10        800.0
...             ...      ...          ...
79283       c_15275  app_997        400.0
109811      c_21194  app_997        400.0
26474        c_5101  app_998        400.0
65555       c_12641  app_998        400.0
28018        c_5402  app_998        400.0

[21103 rows x 3 columns] 3200.0


In [12]:
import csv
import numpy as np
def run(filename,cpu_request):
    chunk = pd.read_csv('/disk7T/vis/code/container/' + str(filename) + '.txt',header=None, sep='\t', encoding="utf-8", quoting=csv.QUOTE_NONE, escapechar=',',names=['time_stamp', 'cpu_util_percent', 'mem_util_percent'])
    if chunk['cpu_util_percent'].sum() < 150000:
        return
    min_time = chunk['time_stamp'].min()
    chunk['time_stamp'] = chunk['time_stamp'] - min_time
    assistance = np.array([i*10 for i in range(69120)])
    assistance = pd.DataFrame(assistance, columns=['time_stamp'])
    chunk = pd.concat([chunk, assistance])
    chunk.drop_duplicates(subset=['time_stamp'], keep='first',inplace=True)
    chunk.sort_values(by='time_stamp', ascending=True,inplace=True)
    chunk['cpu_util_percent'].interpolate(method='linear', inplace=True)
    chunk['mem_util_percent'].interpolate(method='linear', inplace=True)
    chunk['cpu_util_percent'] = chunk['cpu_util_percent'] * cpu_request / 10000
    print(chunk)
    return

chunk = run('c_21194',400)

       time_stamp  cpu_util_percent  mem_util_percent
32350           0              1.40              99.0
61251          10              1.60              99.0
21993          20              1.44              99.0
39553          30              1.36              99.0
13618          40              1.48              99.0
...           ...               ...               ...
29264      691150              1.84             100.0
32349      691160              1.52             100.0
50805      691170              1.64             100.0
2093       691180              1.56             100.0
54029      691190              1.80             100.0

[69120 rows x 3 columns]


In [ ]:
import matplotlib.pyplot as plt
chunk.plot(x='time_stamp',y='cpu_util_percent',kind='line')
plt.title('cpu usage')
plt.xlabel('time')
plt.ylabel('cpu_util_percent')
plt.show()

In [10]:
import pandas as pd
import csv
chunk = pd.read_csv('/disk7T/vis/code/OneMore/data_src/container.txt',header=None, sep='\t', encoding="utf-8", quoting=csv.QUOTE_NONE, escapechar=',')
print(chunk)

       0         1         2         3         4         5         6     \
0       8.0  0.240000  0.400000  0.880000  0.560000  0.240000  0.480000   
1       4.0  0.120000  0.200000  0.120000  0.120000  0.080000  0.120000   
2       4.0  0.360000  0.440000  0.400000  0.320000  0.600000  0.440000   
3       8.0  1.120000  1.840000  1.920000  1.840000  2.400000  1.120000   
4       4.0  0.120000  0.080000  0.120000  0.080000  0.280000  0.120000   
...     ...       ...       ...       ...       ...       ...       ...   
30301   4.0  0.120000  0.280000  0.120000  0.120000  0.160000  0.140000   
30302   4.0  1.680000  1.880000  1.480000  1.400000  1.520000  1.520000   
30303   4.0  1.640000  1.840000  1.560000  1.560000  1.520000  1.640000   
30304   4.0  0.720000  0.740000  0.760000  0.740000  0.720000  0.680000   
30305   4.0  1.838279  1.837439  1.836598  1.835758  1.834917  1.834077   

           7         8         9     ...  8631  8632  8633  8634  8635  8636  \
0      0.400000  0.

In [11]:
import numpy as np
chunk = chunk.values.astype(np.float64)
print(chunk)

[[8.         0.24       0.4        ... 0.8        0.56       0.48      ]
 [4.         0.12       0.2        ... 0.08       0.12       0.12      ]
 [4.         0.36       0.44       ... 0.24       0.28       0.24      ]
 ...
 [4.         1.64       1.84       ... 1.72       1.68       1.68      ]
 [4.         0.72       0.74       ... 0.64       0.68       0.64      ]
 [4.         1.83827914 1.83743872 ... 1.8        1.8        1.8       ]]
